# Bias-SINEX — code measurements have hardware biases

**Goal:** see how an observable-specific bias (**OSB**) affects a GNSS pseudorange, and use `dsoclasses.gnss.biasx.Bsinex` to read a satellite's code bias.

Different tracking signals (`C1W`, `C2W`, etc.) can have different equipment delays. They may be expressed in **nanoseconds** in Bias-SINEX; multiplied by the speed of light, even one nanosecond corresponds to almost **30 cm**.

\[\Delta P\;[\mathrm m]=c\;b\;[\mathrm{ns}]\,10^{-9}.\]

The bundled two-day file is **SYNTHETIC**. It illustrates file parsing and epoch selection, not real G21 biases or compatibility with a particular satellite clock product.

In [ ]:
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from dsoclasses.gnss.biasx import Bsinex
from dsoclasses.gnss import systems as gs

from pathlib import Path

def teaching_data(filename):
    """Find a bundled example with common notebook launch directories."""
    # Jupyter can launch notebooks from its own folder OR project root.
    candidates = []
    for root in (Path.cwd(), *list(Path.cwd().parents)[:4]):
        candidates.extend((root / 'data' / filename, root / filename))
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f'Cannot locate {filename}. Extract the whole lesson ZIP, '
                            'keeping its data/ folder beside notebooks/.')

path = teaching_data('bias_demo.BIA')
reader = Bsinex(path)
when = datetime(2024,1,1,12)
osb = reader.sat_bias('G21', when)  # validity interval selected by epoch
print('Satellite:', 'G21', 'valid from:', osb['from'], 'until:', osb['to'])
for observable in ('C1W','C2W'):
    b_ns = osb[observable]
    print(f'{observable}: {b_ns:+.3f} ns = {b_ns*1e-9*gs.C:+.3f} m')

## 1. Apply the correction before combining measurements

For this lesson we use the same algebraic convention as the earlier GPS positioning notebook:

\[P_{1,\mathrm{corrected}}=P_{1,\mathrm{raw}}-c\,b_1,\qquad
  P_{2,\mathrm{corrected}}=P_{2,\mathrm{raw}}-c\,b_2.\]

The **sign and reference convention depend on the chosen product**: biases and satellite clocks must be mutually consistent. A bias is not a universal number to add to every observation type.

In [ ]:
# Invent a common geometric range and additive hardware biases.
true_range = 23_000_000.0
b1_m = osb['C1W'] * 1e-9 * gs.C
b2_m = osb['C2W'] * 1e-9 * gs.C
p1_raw = true_range + b1_m
p2_raw = true_range + b2_m
p1_fixed = p1_raw - b1_m
p2_fixed = p2_raw - b2_m
print('Raw pseudorange minus assumed true range [m]:',
      f'C1W={p1_raw-true_range:+.3f}, C2W={p2_raw-true_range:+.3f}')
print('Corrected pseudorange minus true range [m]:',
      f'C1W={p1_fixed-true_range:+.6f}, C2W={p2_fixed-true_range:+.6f}')
assert abs(p1_fixed-true_range) < 1e-7
assert abs(p2_fixed-true_range) < 1e-7

## 2. Biases have validity intervals

Your processing epoch matters. If a file contains more than one interval, `sat_bias('G21')` without an epoch is ambiguous and should raise an error. This prevents silently using a stale satellite bias.

In [ ]:
dates=[datetime(2024,1,1,12),datetime(2024,1,2,12)]
for day in dates:
    result = reader.sat_bias('G21', day)
    print(f'{day.date()}: C1W={result["C1W"]:+.2f} ns, '
          f'C2W={result["C2W"]:+.2f} ns')
try:
    reader.sat_bias('G21')
except ValueError as exc:
    print('Expected ambiguity check:', exc)

fig, ax = plt.subplots(figsize=(7.0, 3.6))
for code in ('C1W','C2W'):
    ax.plot([d.date() for d in dates],
            [reader.sat_bias('G21',d)[code]*1e-9*gs.C for d in dates],
            'o-', label=code)
ax.set(xlabel='Validity date', ylabel='Synthetic code bias [m]',
       title='Observable-dependent, epoch-dependent code biases')
ax.grid(alpha=.3)
ax.legend()
plt.show()

### Remember

- Use the **exact code observable** from the RINEX file (`C1W` is not interchangeable with `C1C`).
- Select the correct satellite, epoch and clock/bias convention. `Bsinex.sat_bias(sat, epoch)` returns **nanoseconds**, not metres.
- The ionosphere-free combination eliminates first-order ionospheric refraction, **not** hardware biases. OSB corrections remain relevant.

**Question:** What is a 2 ns pseudorange bias in metres?

Reference: [IGS Bias-SINEX format](https://igs.org/formats-and-standards/).